# 28. Chap2-2 최종 리포트와 SegFormer 분석 인계

앞 노트북의 산출물을 모아 Chapter 2-2 최종 리포트를 생성합니다.

이 리포트는 다음 판단을 내립니다.

- 보정 데이터셋이 타당했는가
- matched condition에서 color/defect/shape 효과가 어떻게 나타났는가
- exposure ratio가 clean transfer를 만들었는가
- 어떤 개선 전략이 최악 조합을 완화했는가
- 이제 SegFormer 내부 아키텍처 분석으로 넘어갈 수 있는가

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-2장/ch2_2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2*/chap2_2/ch2_2_utils.py"))
        + list(Path.cwd().glob("**/ch2_2_utils.py"))
    )
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "2-2장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch2_2_utils import *

paths = find_ch2_2_paths()
set_korean_font()
set_seed(7)
paths

Chapter22Paths(chap2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), chapter2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-1장'), chapter1_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/1장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/data/synthetic_metal_matched'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs/manifests'))

## 28-1. 결과 파일 로드

In [2]:
required = {
    "audit": paths.runs_root / "audit" / "audit_summary.json",
    "baseline_summary": paths.runs_root / "baseline_seed_summary.csv",
    "factor_tests": paths.runs_root / "matched_factor_tests.csv",
    "factor_report": paths.runs_root / "chap2_2_factor_analysis_report.md",
    "exposure_delta": paths.runs_root / "clean_exposure_ratio_delta_summary.csv",
    "strategy": paths.runs_root / "strategy_comparison_summary.csv",
}
missing = [name for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing files. Run previous notebooks first: {missing}")

audit = json.loads(required["audit"].read_text(encoding="utf-8"))
baseline = pd.read_csv(required["baseline_summary"])
factor_tests = pd.read_csv(required["factor_tests"])
factor_report = required["factor_report"].read_text(encoding="utf-8")
exposure_delta = pd.read_csv(required["exposure_delta"])
strategy = pd.read_csv(required["strategy"])
display(baseline)
display(factor_tests)
display(exposure_delta)
display(strategy)

,eval_name,n_rows,n_seeds,mean_target_dice,mean_target_fnr,worst_combo_dice
0,eval_matched,720,3,0.531328,0.501806,0.0
1,eval_stress,480,3,0.000000,1.000000,0.0


,group_col,group_a,group_b,metric,n_seeds,mean_a,mean_b,diff_a_minus_b,ci95_low,ci95_high,reject_h0_ci_excludes_0
0,color_group,red,neutral,target_dice,3,0.057178,0.678630,-0.621452,-0.643988,-0.578800,True
1,color_group,purple,neutral,target_dice,3,0.579481,0.678630,-0.099149,-0.165319,-0.033328,True
2,color_group,red,blue,target_dice,3,0.057178,0.669172,-0.611995,-0.658710,-0.545004,True
3,color_group,purple,blue,target_dice,3,0.579481,0.669172,-0.089691,-0.182460,0.000468,False
4,defect_type,scratch,impact,target_dice,3,0.350144,0.572671,-0.222528,-0.249233,-0.177778,True
5,defect_type,scratch,stain,target_dice,3,0.350144,0.563305,-0.213161,-0.237775,-0.190215,True
6,defect_type,dent,impact,target_dice,3,0.639192,0.572671,0.066521,-0.000686,0.117605,False
7,shape_group,bottom_half_metal,top_half_metal,target_dice,3,0.531225,0.531431,-0.000205,-0.012593,0.018414,False


,eval_combo,dice_at_min_ratio,dice_at_max_ratio,delta_max_minus_min,ci95_low,ci95_high,n_seeds
0,red_scratch,0.197019,0.747547,0.550528,0.513226,0.616881,3
1,purple_scratch,0.358276,0.454675,0.096399,-0.068304,0.189126,3
2,red_impact,0.062914,0.030905,-0.032009,-0.061772,0.000000,3
3,neutral_scratch,0.407495,0.341520,-0.065974,-0.177124,0.014045,3
4,blue_scratch,0.548503,0.384226,-0.164277,-0.249918,-0.035474,3
5,blue_impact,0.727344,0.404418,-0.322926,-0.362232,-0.275517,3
6,neutral_impact,0.689773,0.299918,-0.389855,-0.396488,-0.384710,3
7,purple_impact,0.727600,0.184924,-0.542677,-0.812460,-0.323272,3


,strategy,matched_mean_dice,heldout_color_dice,worst_combo_dice,stress_mean_dice,n_matched_rows,n_stress_rows
0,photometric_aug,0.584793,0.533107,0.111026,0.022616,720,480
1,group_balanced,0.549414,0.353908,0.056238,0.025052,720,480
2,baseline_no_aug,0.531328,0.318329,0.000000,0.000000,720,480


## 28-2. 최종 리포트 작성

In [3]:
matched_row = baseline[baseline["eval_name"] == "eval_matched"].iloc[0]
stress_row = baseline[baseline["eval_name"] == "eval_stress"].iloc[0]
best_strategy = strategy.iloc[0]
exposure_best = exposure_delta.sort_values("delta_max_minus_min", ascending=False).iloc[0]
exposure_worst = exposure_delta.sort_values("delta_max_minus_min").iloc[0]

can_handoff = bool(audit["overall_pass"]) and int(matched_row["n_seeds"]) >= 3

lines = [
    "# Chapter 2-2 최종 리포트",
    "",
    "## 1. 타당성 판정",
    f"- 데이터셋 audit overall_pass: {audit['overall_pass']}",
    f"- train/eval image overlap: {audit['train_eval_image_overlap']}",
    f"- matched baseline model seeds: {int(matched_row['n_seeds'])}",
    f"- SegFormer 내부 분석 인계 가능 여부: {can_handoff}",
    "",
    "## 2. Baseline",
    f"- eval_matched mean Dice: {matched_row['mean_target_dice']:.3f}",
    f"- eval_matched worst combo Dice: {matched_row['worst_combo_dice']:.3f}",
    f"- eval_stress mean Dice: {stress_row['mean_target_dice']:.3f}",
    f"- eval_stress worst combo Dice: {stress_row['worst_combo_dice']:.3f}",
    "",
    "## 3. Factor 검정",
    "```text",
    factor_tests.to_string(index=False),
    "```",
    "",
    "## 4. Clean Exposure Ratio",
    f"- 가장 좋아진 eval combo: {exposure_best['eval_combo']} delta={exposure_best['delta_max_minus_min']:.3f}",
    f"- 가장 나빠진 eval combo: {exposure_worst['eval_combo']} delta={exposure_worst['delta_max_minus_min']:.3f}",
    "",
    "## 5. 개선 전략",
    f"- best strategy by sorted worst/heldout criteria: {best_strategy['strategy']}",
    f"- matched_mean_dice: {best_strategy['matched_mean_dice']:.3f}",
    f"- heldout_color_dice: {best_strategy['heldout_color_dice']:.3f}",
    f"- worst_combo_dice: {best_strategy['worst_combo_dice']:.3f}",
    f"- stress_mean_dice: {best_strategy['stress_mean_dice']:.3f}",
    "",
    "## 6. 다음 단계",
    "데이터셋 audit와 seed 반복이 통과했으므로, 위 factor 검정에서 실제로 유의한 실패 축만 SegFormer 아키텍처 분석 대상으로 넘긴다.",
]
report_path = paths.runs_root / "chapter2_2_final_report.md"
report_path.write_text("\n".join(lines), encoding="utf-8")
print(report_path)
print("\n".join(lines[:30]))

C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\chapter2_2_final_report.md
# Chapter 2-2 최종 리포트

## 1. 타당성 판정
- 데이터셋 audit overall_pass: True
- train/eval image overlap: 0
- matched baseline model seeds: 3
- SegFormer 내부 분석 인계 가능 여부: True

## 2. Baseline
- eval_matched mean Dice: 0.531
- eval_matched worst combo Dice: 0.000
- eval_stress mean Dice: 0.000
- eval_stress worst combo Dice: 0.000

## 3. Factor 검정
```text
  group_col           group_a        group_b      metric  n_seeds   mean_a   mean_b  diff_a_minus_b  ci95_low  ci95_high  reject_h0_ci_excludes_0
color_group               red        neutral target_dice        3 0.057178 0.678630       -0.621452 -0.643988  -0.578800                     True
color_group            purple        neutral target_dice        3 0.579481 0.678630       -0.099149 -0.165319  -0.033328                     True
color_group               red           blue target_dice        3 0.057178 0.669172       -0.611995 -0.658710  -0.545004    

## 28-3. SegFormer 아키텍처 분석 인계 계획

In [4]:
handoff = '''# SegFormer 아키텍처 분석 인계 계획

## 전제

Chapter 2-2의 matched factorial audit가 통과하고, seed 반복 결과에서 유의한 실패 축만 내부 분석 대상으로 삼는다.

## 파트별 가설

| 파트 | 가설 | 검증 실험 | 해법 후보 |
|---|---|---|---|
| Input normalization / RGB stem | heldout color에서 early feature가 ImageNet RGB prior에 끌린다 | RGB histogram matching, grayscale eval, first patch embedding PCA | per-domain normalization, color jitter, grayscale/color-drop |
| Overlap patch embedding | scratch 같은 얇은 결함이 stride 4 stem에서 약해진다 | scratch width별 Dice, 128 vs 224 input 비교 | higher resolution, smaller stride stem |
| Efficient self-attention SR | small defect token이 spatial reduction에서 희석된다 | SR ratio ablation, attention rollout | lower SR ratio, crop training |
| Mix-FFN depthwise conv | local texture/color shortcut을 학습한다 | texture randomization, feature invariance | style randomization, stronger photometric aug |
| MLP decoder fusion | high-res boundary 정보가 fusion에서 손실된다 | decoder feature ablation, boundary F1 | aux high-res head, boundary loss |
| Final classifier/loss | defect pixel imbalance로 FNR이 높아진다 | gradient norm, loss ablation | Dice+CE, Focal/Tversky, hard mining |
'''
            handoff_path = paths.runs_root / "segformer_architecture_handoff_plan.md"
            handoff_path.write_text(handoff, encoding="utf-8")
            print(handoff_path)

IndentationError: unexpected indent (864914442.py, line 18)